# Knowledge Base Setup, End to End on Bedrock (S3 Vectors)

**Track:** Agentic AI Bootcamp &nbsp;|&nbsp; **Level:** Advanced

This notebook **creates a real, queryable Bedrock Knowledge Base** and prints a `KB_ID` you paste into notebook 02. It uses **Amazon S3 Vectors** as the vector store, which provisions in seconds (no OpenSearch collection to wait on), so it runs live in class.

Run every cell top to bottom once. The teardown at the end removes everything so you are not billed after the demo.

## What gets built

```mermaid
flowchart TD
    A[Upload policy docs to an S3 source bucket] --> B[Create an S3 vector bucket + vector index]
    B --> C[Create an IAM role Bedrock can assume]
    C --> D[Create the Knowledge Base pointing at the vector index]
    D --> E[Attach the S3 data source]
    E --> F[Run an ingestion job: chunk, embed, index]
    F --> G[Retrieve + RetrieveAndGenerate = RAG]
    G --> H[Copy KB_ID into notebook 02]
```

## The three gotchas this notebook handles for you

| Gotcha | What breaks without it | Handled by |
|---|---|---|
| boto3 too old | `Unknown service: 's3vectors'` | Version check in section 0 |
| Titan embeddings not enabled | `AccessDeniedException` on first ingest | Section 0 reminder + preflight |
| Filterable-metadata 2 KB limit | Ingestion fails: text chunk overflows 2 KB | `nonFilterableMetadataKeys` set at index creation (section 2) |


## 0. Setup and preflight

**VS Code:** select your venv as the kernel, `aws configure` (region **us-east-1**), then run the install cell.
**Colab:** run the install cell, set credentials via Colab secrets or env vars.

**Before you run anything, enable the embedding model once:** Bedrock console (us-east-1) → Model access → enable **Titan Text Embeddings V2** (`amazon.titan-embed-text-v2:0`). Without it, ingestion fails with `AccessDeniedException` on the model ARN.


In [ ]:
%pip install -q --upgrade "boto3>=1.39.9" botocore

In [ ]:
import json, time, uuid
import boto3, botocore

# S3 Vectors needs a recent SDK. Fail loudly and early if it is too old.
print("boto3:", boto3.__version__, "| botocore:", botocore.__version__)
assert tuple(int(x) for x in boto3.__version__.split(".")[:3]) >= (1, 39, 9), \
    "Upgrade boto3 to 1.39.9+ (the s3vectors client is not in older versions)."

REGION = "us-east-1"
sts = boto3.client("sts", region_name=REGION)
ACCOUNT = sts.get_caller_identity()["Account"]

EMBED_MODEL_ID = "amazon.titan-embed-text-v2:0"
EMBED_MODEL_ARN = f"arn:aws:bedrock:{REGION}::foundation-model/{EMBED_MODEL_ID}"
EMBED_DIM = 1024                       # Titan v2 default; MUST match the index dimension
GEN_MODEL_ARN = f"arn:aws:bedrock:{REGION}::foundation-model/us.anthropic.claude-haiku-4-5-20251001-v1:0"

# unique names so re-runs and multiple students do not collide
SUFFIX     = uuid.uuid4().hex[:8]
SRC_BUCKET = f"travelmind-kb-src-{ACCOUNT}-{SUFFIX}"
VEC_BUCKET = f"travelmind-kb-vec-{ACCOUNT}-{SUFFIX}"
VEC_INDEX  = "travelmind-index"
KB_NAME    = f"travelmind-kb-{SUFFIX}"
ROLE_NAME  = f"travelmind-kb-role-{SUFFIX}"

print("account:", ACCOUNT, "| region:", REGION)
print("source bucket:", SRC_BUCKET)
print("vector bucket:", VEC_BUCKET, "| index:", VEC_INDEX)


In [ ]:
# Preflight: confirm the embedding model is actually invokable before we build anything.
# Cheaper to fail here than three minutes into ingestion.
try:
    brt = boto3.client("bedrock-runtime", region_name=REGION)
    r = brt.invoke_model(modelId=EMBED_MODEL_ID,
                         body=json.dumps({"inputText": "preflight"}))
    dim = len(json.loads(r["body"].read())["embedding"])
    print(f"Titan embeddings OK, dimension = {dim}")
    assert dim == EMBED_DIM, f"Model dim {dim} != EMBED_DIM {EMBED_DIM}; fix EMBED_DIM."
except botocore.exceptions.ClientError as e:
    raise SystemExit("Enable 'Titan Text Embeddings V2' in the Bedrock console (us-east-1) "
                     "under Model access, then re-run. Error: " + str(e))


## 1. Create the source bucket and upload the corpus

The corpus is a few small TravelMind airline policy documents. Retrieval later pulls from these, so the agent answers from policy instead of guessing.

In [ ]:
s3 = boto3.client("s3", region_name=REGION)

# us-east-1 is special: create_bucket must NOT pass LocationConstraint
if REGION == "us-east-1":
    s3.create_bucket(Bucket=SRC_BUCKET)
else:
    s3.create_bucket(Bucket=SRC_BUCKET,
                     CreateBucketConfiguration={"LocationConstraint": REGION})

DOCS = {
    "refund_policy.txt": (
        "TravelMind Refund Policy. When a flight is cancelled by the airline, the passenger "
        "is entitled to a full refund of base fare and taxes. Gold tier members receive an "
        "additional 10 percent loyalty credit on the refunded base fare. Refunds are issued "
        "to the original form of payment within 7 business days. Vouchers are optional and "
        "never mandatory."
    ),
    "rebooking_policy.txt": (
        "TravelMind Rebooking Policy. On an airline-initiated cancellation, rebooking on the "
        "next available flight is free of charge, including fare difference within the same "
        "cabin. Gold tier members may rebook into a higher cabin once per disruption at no "
        "extra cost, subject to availability. Rebooking must be completed within 72 hours of "
        "the original departure to remain free."
    ),
    "tier_benefits.txt": (
        "TravelMind Loyalty Tiers. Silver, Gold, and Platinum. Gold benefits include priority "
        "rebooking, a 10 percent refund loyalty credit on disruptions, one free cabin upgrade "
        "per disruption, and waived change fees. Platinum adds lounge access and guaranteed "
        "seat availability on full flights."
    ),
    "cancellation_windows.txt": (
        "TravelMind Cancellation Windows. Passenger-initiated cancellations more than 24 hours "
        "before departure incur no fee for refundable fares. Non-refundable fares receive travel "
        "credit minus a service fee. Airline-initiated cancellations always qualify for a full "
        "refund regardless of fare type."
    ),
}
for name, text in DOCS.items():
    s3.put_object(Bucket=SRC_BUCKET, Key=name, Body=text.encode("utf-8"))
print(f"uploaded {len(DOCS)} docs to s3://{SRC_BUCKET}/")


## 2. Create the S3 vector bucket and index

This is the vector store. **The critical line is `nonFilterableMetadataKeys`.** Bedrock stores the chunk text under `AMAZON_BEDROCK_TEXT` and document metadata under `AMAZON_BEDROCK_METADATA`. S3 Vectors caps *filterable* metadata at 2 KB per vector, and a normal text chunk blows past that. Declaring both keys non-filterable raises the ceiling to 40 KB and lets ingestion succeed.

Index settings are permanent (dimension, distance metric, non-filterable keys). Get them right the first time.

In [ ]:
s3v = boto3.client("s3vectors", region_name=REGION)

s3v.create_vector_bucket(vectorBucketName=VEC_BUCKET)

s3v.create_index(
    vectorBucketName=VEC_BUCKET,
    indexName=VEC_INDEX,
    dataType="float32",              # S3 Vectors supports float32 only (no binary)
    dimension=EMBED_DIM,             # must equal the embedding model's output dimension
    distanceMetric="cosine",
    metadataConfiguration={
        "nonFilterableMetadataKeys": ["AMAZON_BEDROCK_TEXT", "AMAZON_BEDROCK_METADATA"]
    },
)

# ARNs follow a fixed pattern; build them directly (robust across SDK response shapes)
VEC_BUCKET_ARN = f"arn:aws:s3vectors:{REGION}:{ACCOUNT}:bucket/{VEC_BUCKET}"
VEC_INDEX_ARN  = f"{VEC_BUCKET_ARN}/index/{VEC_INDEX}"
print("vector index ready:", VEC_INDEX_ARN)


## 3. Create the IAM role Bedrock assumes

The Knowledge Base is a service that acts on your behalf. It needs a role it can assume with three permissions: invoke the embedding model, read the source bucket, and read/write the vector index. The trust policy pins it to your account and to knowledge-base resources so nothing else can assume it.

In [ ]:
iam = boto3.client("iam")

trust = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock.amazonaws.com"},
        "Action": "sts:AssumeRole",
        "Condition": {
            "StringEquals": {"aws:SourceAccount": ACCOUNT},
            "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock:{REGION}:{ACCOUNT}:knowledge-base/*"}
        }
    }]
}
role = iam.create_role(RoleName=ROLE_NAME,
                       AssumeRolePolicyDocument=json.dumps(trust),
                       Description="TravelMind KB service role")
ROLE_ARN = role["Role"]["Arn"]

policy = {
    "Version": "2012-10-17",
    "Statement": [
        {"Sid": "InvokeEmbedModel", "Effect": "Allow",
         "Action": ["bedrock:InvokeModel"], "Resource": [EMBED_MODEL_ARN]},
        {"Sid": "ReadSourceBucket", "Effect": "Allow",
         "Action": ["s3:ListBucket", "s3:GetObject"],
         "Resource": [f"arn:aws:s3:::{SRC_BUCKET}", f"arn:aws:s3:::{SRC_BUCKET}/*"]},
        {"Sid": "S3Vectors", "Effect": "Allow",
         "Action": ["s3vectors:PutVectors", "s3vectors:GetVectors",
                    "s3vectors:QueryVectors", "s3vectors:DeleteVectors",
                    "s3vectors:GetIndex", "s3vectors:ListVectors"],
         "Resource": [VEC_INDEX_ARN]},
    ],
}
iam.put_role_policy(RoleName=ROLE_NAME, PolicyName="kb-inline",
                    PolicyDocument=json.dumps(policy))
print("role:", ROLE_ARN)

# IAM is eventually consistent. A fresh role is not always assumable immediately,
# so give it a moment before the KB tries to assume it.
print("waiting 15s for IAM propagation...")
time.sleep(15)


## 4. Create the Knowledge Base

Point it at the embedding model and the vector index. `embeddingDataType` must be `FLOAT32` and `dimensions` must match the index (1024). Retry on `AccessDenied` in case the role is still propagating.

In [ ]:
bagent = boto3.client("bedrock-agent", region_name=REGION)

def create_kb_with_retry(attempts=5):
    for i in range(attempts):
        try:
            return bagent.create_knowledge_base(
                name=KB_NAME,
                description="TravelMind airline policy KB",
                roleArn=ROLE_ARN,
                knowledgeBaseConfiguration={
                    "type": "VECTOR",
                    "vectorKnowledgeBaseConfiguration": {
                        "embeddingModelArn": EMBED_MODEL_ARN,
                        "embeddingModelConfiguration": {
                            "bedrockEmbeddingModelConfiguration": {
                                "dimensions": EMBED_DIM,
                                "embeddingDataType": "FLOAT32",
                            }
                        },
                    },
                },
                storageConfiguration={
                    "type": "S3_VECTORS",
                    "s3VectorsConfiguration": {
                        "vectorBucketArn": VEC_BUCKET_ARN,
                        "indexArn": VEC_INDEX_ARN,
                    },
                },
            )
        except botocore.exceptions.ClientError as e:
            code_ = e.response["Error"]["Code"]
            if code_ in ("AccessDeniedException", "ValidationException") and i < attempts - 1:
                print(f"  role not ready ({code_}); retry {i+1} in 10s")
                time.sleep(10)
            else:
                raise

kb = create_kb_with_retry()
KB_ID = kb["knowledgeBase"]["knowledgeBaseId"]
print("knowledge base:", KB_ID)


## 5. Attach the S3 data source and run ingestion

The data source tells the KB where the documents live. The ingestion job is the actual work: crawl, chunk (default fixed-size ~300 tokens), embed with Titan, write vectors to the index. Poll until `COMPLETE`. For four tiny docs this is under a minute.

In [ ]:
ds = bagent.create_data_source(
    knowledgeBaseId=KB_ID,
    name="travelmind-s3-source",
    dataSourceConfiguration={
        "type": "S3",
        "s3Configuration": {"bucketArn": f"arn:aws:s3:::{SRC_BUCKET}"},
    },
)
DS_ID = ds["dataSource"]["dataSourceId"]
print("data source:", DS_ID)

job = bagent.start_ingestion_job(knowledgeBaseId=KB_ID, dataSourceId=DS_ID)
JOB_ID = job["ingestionJob"]["ingestionJobId"]
print("ingestion job:", JOB_ID, "- polling...")

while True:
    st = bagent.get_ingestion_job(knowledgeBaseId=KB_ID, dataSourceId=DS_ID,
                                  ingestionJobId=JOB_ID)["ingestionJob"]["status"]
    print("  status:", st)
    if st in ("COMPLETE", "FAILED"):
        break
    time.sleep(10)

assert st == "COMPLETE", "Ingestion failed. Check the job's failureReasons and IAM policy."
print("ingestion complete")


## 6. Use it: Retrieve and RetrieveAndGenerate

Two runtime operations on `bedrock-agent-runtime`:
- **retrieve** returns raw matching chunks (you own the prompt after).
- **retrieve_and_generate** does full RAG: retrieve, stuff into a prompt, generate a grounded answer with citations.

In [ ]:
brt_agent = boto3.client("bedrock-agent-runtime", region_name=REGION)

# retrieve: raw chunks
q = "What refund does a Gold tier passenger get when the airline cancels the flight?"
res = brt_agent.retrieve(
    knowledgeBaseId=KB_ID,
    retrievalQuery={"text": q},
    retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 3}},
)
print("Top chunks:")
for i, r in enumerate(res["retrievalResults"], 1):
    print(f"  [{i}] score={r.get('score'):.3f}  {r['content']['text'][:120]}...")


In [ ]:
# retrieve_and_generate: grounded answer with citations
rag = brt_agent.retrieve_and_generate(
    input={"text": q},
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            "knowledgeBaseId": KB_ID,
            "modelArn": GEN_MODEL_ARN,
        },
    },
)
print("Answer:\n", rag["output"]["text"])
print("\nCitations:", len(rag.get("citations", [])))


## 7. Copy this into notebook 02

The value below is what the KB section of notebook 02 expects. Paste it into the `KB_ID = ...` line there.

In [ ]:
print("=" * 52)
print("  KB_ID =", KB_ID)
print("  (region:", REGION, ")")
print("=" * 52)


## 8. Teardown (run after the demo to stop billing)

Order matters: data source and KB first, then the vector index and buckets, then the role. Skip this only if you want to keep the KB for notebook 02 across sessions (it keeps costing a little while it exists).

In [ ]:
def _try(label, fn):
    try:
        fn(); print("deleted", label)
    except Exception as e:
        print(label, "->", e)

_try("data source",   lambda: bagent.delete_data_source(knowledgeBaseId=KB_ID, dataSourceId=DS_ID))
time.sleep(3)
_try("knowledge base", lambda: bagent.delete_knowledge_base(knowledgeBaseId=KB_ID))
time.sleep(3)
_try("vector index",  lambda: s3v.delete_index(vectorBucketName=VEC_BUCKET, indexName=VEC_INDEX))
_try("vector bucket", lambda: s3v.delete_vector_bucket(vectorBucketName=VEC_BUCKET))

# empty then delete the source bucket
def _empty_src():
    objs = s3.list_objects_v2(Bucket=SRC_BUCKET).get("Contents", [])
    for o in objs:
        s3.delete_object(Bucket=SRC_BUCKET, Key=o["Key"])
    s3.delete_bucket(Bucket=SRC_BUCKET)
_try("source bucket", _empty_src)

def _del_role():
    iam.delete_role_policy(RoleName=ROLE_NAME, PolicyName="kb-inline")
    iam.delete_role(RoleName=ROLE_NAME)
_try("iam role", _del_role)
print("teardown done")


**What changes in production**
- Role via IaC (CloudFormation/CDK/Terraform), least-privilege, not created ad hoc in a notebook.
- Real corpus in S3 with a sync schedule; re-run ingestion on document changes (incremental).
- Tune chunking for your documents; fixed-size 300 is a starting point, not an answer.
- Add a guardrail to `retrieve_and_generate` via `generationConfiguration.guardrailConfiguration`.
- Keep the KB alive (do not tear down); manage its lifecycle separately from demos.